## Informational 담당

In [1]:
import pandas as pd
all_merge_df = pd.read_csv('../data/all_merged.csv')

- 정상적인 Informational로 피벗테이블 만들기

In [2]:
# informational만 뽑기
info_df = all_merge_df[all_merge_df['offer_type'] == 'informational'].copy()
# offer_cycle 만들기
info_df = info_df.sort_values(['person', 'offer_id', 'time'])
info_df['is_received'] = (info_df['event'] == 'offer received').astype(int)
info_df['offer_cycle'] = info_df.groupby(['person', 'offer_id'])['is_received'].cumsum()

# 피벗 테이블
info_pivot = info_df.pivot_table(
    index=['person', 'offer_id', 'offer_cycle', 'duration'], # duration 추가
    columns='event', values='time', aggfunc='min'
).reset_index()

# expire_time : 유효기간 /// duration은 일(days)단위, 시간(hours)과 맞추기
info_pivot['expire_time'] = info_pivot['offer received'] + (info_pivot['duration'] * 24)

# 결제 정보 가져오기
all_transactions = all_merge_df[all_merge_df['event'] == 'transaction'][['person', 'time', 'amount']]

# 모든 transaction을 person을 기준으로 left join
info_trans_merge = pd.merge(info_pivot, all_transactions, on='person', how='left')

print(f"left join 후 행 크기 : {len(info_trans_merge)}")
# 필터링: 유효기간 내에 결제한 것만 추리기
success_mask = (
    info_trans_merge['offer viewed'].notna() &
    (info_trans_merge['time'] >= info_trans_merge['offer viewed']) &
    (info_trans_merge['time'] <= info_trans_merge['expire_time'])
)
valid_info_transactions = info_trans_merge[success_mask].copy()
print(f"유효기간 내 결제 건 수 : {len(valid_info_transactions)}")
display(valid_info_transactions)

left join 후 행 크기 : 121643
유효기간 내 결제 건 수 : 10612


,person,offer_id,offer_cycle,duration,offer received,offer viewed,expire_time,time,amount
0,0009655768c64bdeb2e877511632db8f,informational_0_0_3,1,3.0,168.0,192.0,240.0,228.0,22.16
9,0009655768c64bdeb2e877511632db8f,informational_0_0_4,1,4.0,336.0,372.0,432.0,414.0,8.57
40,0020ccbbb6d84e358d3414a3ff76cffd,informational_0_0_3,1,3.0,408.0,408.0,480.0,426.0,8.93
41,0020ccbbb6d84e358d3414a3ff76cffd,informational_0_0_3,1,3.0,408.0,408.0,480.0,432.0,20.08
42,0020ccbbb6d84e358d3414a3ff76cffd,informational_0_0_3,1,3.0,408.0,408.0,480.0,450.0,10.76
...,...,...,...,...,...,...,...,...,...
121583,ffeaa02452ef451082a0361c3ca62ef5,informational_0_0_3,2,3.0,408.0,420.0,480.0,420.0,19.99
121603,fff0f0aac6c547b9b263080f09a5586a,informational_0_0_4,2,4.0,576.0,636.0,672.0,660.0,17.10
121624,fff3ba4757bd42088c044ca26d73817a,informational_0_0_3,2,3.0,504.0,540.0,576.0,540.0,388.22
121625,fff3ba4757bd42088c044ca26d73817a,informational_0_0_3,2,3.0,504.0,540.0,576.0,552.0,26.09


- 쿠폰별(개별) 지표

In [3]:
# 건당 얼마 썼는지 계산
instance_revenue = valid_info_transactions.groupby(['person', 'offer_id', 'offer_cycle']).agg(
    total_amount=('amount', 'sum')
).reset_index()

# 최종 지표
info_performance = instance_revenue.groupby('offer_id').agg(
    conversion_count=('person', 'count'), # 마케팅 성공 횟수(전환 횟수)
    total_revenue=('total_amount', 'sum')  # 총 발생 매출 (= 순수익)
).reset_index()

info_performance = info_performance.sort_values(by='total_revenue', ascending=False)

print("Informational 쿠폰별 개별 통합 지표")
display(info_performance)

Informational 쿠폰별 개별 통합 지표


,offer_id,conversion_count,total_revenue
0,informational_0_0_3,3650,73885.68
1,informational_0_0_4,2304,59834.51


- Informational 통합 지표

In [4]:
# informational 자체에 대한 총 매출액 (informational_0_0_3,informational_0_0_4 구분없이!)

# 이 경우는 더블카운팅이 문제가 되지만 동일 person, time, amount에 대한 행들을 하나만 남기고 더하면 해결됨
unique_info_transactions = valid_info_transactions.drop_duplicates(subset=['person', 'time', 'amount'])
real_total_info_revenue = unique_info_transactions['amount'].sum()

print("Informational 전체 통합 지표")
print(f"순수 결제 건수: {len(unique_info_transactions)}건")
print(f"실제 총 발생 매출액: {real_total_info_revenue}")

Informational 전체 통합 지표
순수 결제 건수: 10564건
실제 총 발생 매출액: 132995.07


- Informational 퍼널 및 이탈률 계산

In [5]:
# [1. Informational 통합 퍼널 및 이탈률 계산]

# 1: Received
# - info_pivot의 전체 행 수가 곧, 전체 Received 수
total_received = len(info_pivot)

# 2: Viewed
# - viewed 시간이 존재하고, 받은 시간 이후에 본 정상적인 케이스
viewed_mask = info_pivot['offer viewed'].notna() & (info_pivot['offer viewed'] >= info_pivot['offer received'])
total_viewed = len(info_pivot[viewed_mask])

# 3: Transaction
# 한 사이클(쿠폰 1회 발송) 안에서 여러 번 결제했더라도 퍼널 단계에서는 '1건의 성공'으로 쳐야 하므로 중복 제거!
total_converted = valid_info_transactions[['person', 'offer_id', 'offer_cycle']].drop_duplicates().shape[0]

# 이탈률 계산(%)
drop_rate_received_to_viewed = (1 - (total_viewed / total_received)) * 100
drop_rate_viewed_to_trans = (1 - (total_converted / total_viewed)) * 100

print("[통합] Informational 퍼널 분석")
print(f"1. Received : {total_received}건")
print(f"2. Viewed : {total_viewed}건 (이탈률: {drop_rate_received_to_viewed:.1f}%)")
print(f"3. Transaction : {total_converted}건 (이탈률: {drop_rate_viewed_to_trans:.1f}%)")

print("-" * 80)


# [2. 개별 쿠폰별 퍼널 및 이탈률 계산]

# 각 단계별 offer_id 기준 카운트 집계
coupon_received = info_pivot.groupby('offer_id').size().reset_index(name='received_count')
coupon_viewed = info_pivot[viewed_mask].groupby('offer_id').size().reset_index(name='viewed_count')
coupon_converted = valid_info_transactions[['person', 'offer_id', 'offer_cycle']].drop_duplicates().groupby('offer_id').size().reset_index(name='transaction_count')

# 3가지 케이스 병합 (Left Join)
funnel_df = pd.merge(coupon_received, coupon_viewed, on='offer_id', how='left')
funnel_df = pd.merge(funnel_df, coupon_converted, on='offer_id', how='left')

# 쿠폰별 전환율 계산
#funnel_df['drop_rate(recevied => viewed) (%)'] = ((1 - (funnel_df['viewed_count'] / funnel_df['received_count'])) * 100).round(1) # 이탈률
funnel_df['View_Rate (%)'] = (((funnel_df['viewed_count'] / funnel_df['received_count'])) * 100).round(1) # 전환율
#funnel_df['drop_rate(viewed => transaction)(%)'] = ((1 - (funnel_df['transaction_count'] / funnel_df['viewed_count'])) * 100).round(1) # 이탈률
funnel_df['View_to_Trans_Rate(%)'] = (((funnel_df['transaction_count'] / funnel_df['viewed_count'])) * 100).round(1) # 전환율

print("\n[개별 쿠폰] Informational 퍼널 분석")
display(funnel_df)

[통합] Informational 퍼널 분석
1. Received : 15235건
2. Viewed : 10831건 (이탈률: 28.9%)
3. Transaction : 5954건 (이탈률: 45.0%)
--------------------------------------------------------------------------------

[개별 쿠폰] Informational 퍼널 분석


,offer_id,received_count,viewed_count,transaction_count,View_Rate (%),View_to_Trans_Rate(%)
0,informational_0_0_3,7618,6687,3650,87.8,54.6
1,informational_0_0_4,7617,4144,2304,54.4,55.6


- 고객 정보별 전환율 구하기

In [6]:
# 전환율 구하기

user_profiles = all_merge_df[['person', 'age', 'gender', 'income']].drop_duplicates()

user_profiles['age_group'] = pd.cut(
    user_profiles['age'], 
    bins=[0, 19, 29, 39, 49, 59, 69, 999], 
    labels=['10대 이하', '20대', '30대', '40대', '50대', '60대', '70대 이상']
)

user_profiles['income_group'] = pd.cut(
    user_profiles['income'], 
    bins=[0, 40000, 60000, 80000, 100000, 200000], 
    labels=['4만 미만', '4만~6만', '6만~8만', '8만~10만', '10만 이상']
)


# 결측치 'Unknown'으로
user_profiles['age_group'] = user_profiles['age_group'].astype(str).replace('nan', 'Unknown')
user_profiles['income_group'] = user_profiles['income_group'].astype(str).replace('nan', 'Unknown')


# info_pivot에 고객 정보와 '성공 여부' 붙이기
info_demo = pd.merge(info_pivot, user_profiles, on='person', how='left')

# 조회 여부 (1 or 0)
info_demo['is_viewed'] = (info_demo['offer viewed'].notna() & (info_demo['offer viewed'] >= info_demo['offer received'])).astype(int)

# 전환 여부 (1 or 0)
# 유효기간 내에 결제한 성공 사이클 목록을 가져와서 1로 표시
success_cycles = valid_info_transactions[['person', 'offer_id', 'offer_cycle']].drop_duplicates()
success_cycles['is_converted'] = 1

# 원본에 병합해서 성공한 애들은 1, 결제 안 한 애들은 0으로
info_demo = pd.merge(info_demo, success_cycles, on=['person', 'offer_id', 'offer_cycle'], how='left')
info_demo['is_converted'] = info_demo['is_converted'].fillna(0).astype(int)


# 속성별 퍼널 체크 함수
def get_demo_funnel(df, group_col):
    demo_funnel = df.groupby(group_col).agg(
        Received=('person', 'count'),
        Viewed=('is_viewed', 'sum'),
        Converted=('is_converted', 'sum')
    ).reset_index()
    
    # 핵심 지표 계산
    demo_funnel['View_Rate (%)'] = (demo_funnel['Viewed'] / demo_funnel['Received'] * 100).round(1) # Received -> Viewed
    demo_funnel['View_to_Trans_Rate (%)'] = (demo_funnel['Converted'] / demo_funnel['Viewed'] * 100).round(1) # Viewed -> Transaction
    demo_funnel['Final_Conversion_Rate (%)'] = (demo_funnel['Converted'] / demo_funnel['Received'] * 100).round(1) # Received -> Transaction(최종)
    
    # 'Final_Conversion_Rate'이 높은 순으로 정렬
    demo_funnel = demo_funnel.fillna(0).sort_values('Final_Conversion_Rate (%)', ascending=False).reset_index(drop=True)
    return demo_funnel

# 결과 출력
print(" [1] Gender - Informational 퍼널 분석 ")
display(get_demo_funnel(info_demo, 'gender'))

print("=" * 60)
print(" [2] Age - Informational 퍼널 분석 ")
display(get_demo_funnel(info_demo, 'age_group'))

print("=" * 60)
print(" [3] Income - Informational 퍼널 분석 ")
display(get_demo_funnel(info_demo, 'income_group'))

 [1] Gender - Informational 퍼널 분석 


,gender,Received,Viewed,Converted,View_Rate (%),View_to_Trans_Rate (%),Final_Conversion_Rate (%)
0,O,195,161,84,82.6,52.2,43.1
1,M,7567,5289,3031,69.9,57.3,40.1
2,F,5538,3910,2137,70.6,54.7,38.6
3,Unknown,1935,1471,702,76.0,47.7,36.3


 [2] Age - Informational 퍼널 분석 


,age_group,Received,Viewed,Converted,View_Rate (%),View_to_Trans_Rate (%),Final_Conversion_Rate (%)
0,10대 이하,199,140,90,70.4,64.3,45.2
1,30대,1378,937,611,68.0,65.2,44.3
2,40대,2047,1550,848,75.7,54.7,41.4
3,20대,1177,749,479,63.6,64.0,40.7
4,50대,3202,2304,1259,72.0,54.6,39.3
5,70대 이상,2654,1838,1000,69.3,54.4,37.7
6,60대,2643,1842,965,69.7,52.4,36.5
7,Unknown,1935,1471,702,76.0,47.7,36.3


 [3] Income - Informational 퍼널 분석 


,income_group,Received,Viewed,Converted,View_Rate (%),View_to_Trans_Rate (%),Final_Conversion_Rate (%)
0,4만~6만,4165,2947,1789,70.8,60.7,43.0
1,4만 미만,1879,1202,779,64.0,64.8,41.5
2,6만~8만,4073,3013,1683,74.0,55.9,41.3
3,Unknown,1935,1471,702,76.0,47.7,36.3
4,8만~10만,2294,1700,806,74.1,47.4,35.1
5,10만 이상,889,498,195,56.0,39.2,21.9


In [7]:
info_demo.to_csv('../data/info_demo.csv', index=False)
valid_info_transactions.to_csv('../data/info.csv', index=False)